## 07 - CIBERSORTx Deconvolution (split out from notebook 04's plan)

### Aim
* Run CIBERSORTx Fractions mode (LM22 default signature) on the TCGA-PAAD cohort to get an independent, reference-based immune-cell-fraction estimate to cross-validate against XDec (notebook 06) and TIMER (notebook 08).
* **Kernel note:** Python 3, not this project's usual `ir` kernel - matches notebook 06. CIBERSORTx itself is Docker/language-agnostic infrastructure, so this is a style choice for consistency with the XDec notebook, not a technical requirement the way it was for XDec.

### Input
* `data/processed/TCGA-PAAD_TPM_raw.tsv` (notebook 02) - confirmed non-log, linear TPM, `gene_symbol` row index - matches CIBERSORTx's documented mixture-file format directly, no transformation needed.

### Output
* `data/deconvolution/cibersortx/CIBERSORTx_Results.txt` (per-sample LM22 cell-type fractions) - **not yet produced**, see blocker below.

### Steps
1. Stage the mixture file (copy of the TPM matrix, already in the right format).
2. Run `cibersortx/fractions` via Docker with the LM22 default signature.
3. Load and inspect the results once produced.

### Result
* Step 1 done. Step 2 **blocked**: requires a CIBERSORTx token (register at https://cibersortx.stanford.edu - free, but a manual step only you can do) and Docker Desktop running. Docker itself is confirmed installed on this machine (v27.3.1) - just the token is missing.

### Conclusion
* Ready to run as soon as a token is obtained - substitute it into the `docker run` command below and execute.

In [1]:
import os
import shutil

import pandas as pd

if os.path.basename(os.getcwd()) == "rnaseq_download":
    os.chdir(os.path.join("..", ".."))
os.getcwd()

'/Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer/progress'

### Step 1: stage the mixture file

In [2]:
ciber_dir = "data/deconvolution/cibersortx"
os.makedirs(ciber_dir, exist_ok=True)

mixture_path = os.path.join(ciber_dir, "mixture_TPM.txt")
if not os.path.exists(mixture_path):
    shutil.copy("data/processed/TCGA-PAAD_TPM_raw.tsv", mixture_path)

print("Mixture file ready at:", mixture_path)
with open(mixture_path) as fh:
    print(fh.readline()[:200])

Mixture file ready at: data/deconvolution/cibersortx/mixture_TPM.txt
gene_symbol	TCGA-IB-7888-01A-11R-2156-07	TCGA-FB-A78T-01A-12R-A32O-07	TCGA-2J-AABI-01A-12R-A41B-07	TCGA-HZ-7918-01A-11R-2156-07	TCGA-2J-AABO-01A-21R-A41B-07	TCGA-IB-AAUN-01A-12R-A38C-07	TCGA-HZ-7924-0


### Step 2 (blocked - needs a token): run CIBERSORTx Fractions via Docker

Not executed here. Once you have a token from https://cibersortx.stanford.edu, run this in a terminal (not this notebook - it's a long-running Docker job, not something to block a Jupyter kernel on):

```bash
docker run -v $(pwd)/data/deconvolution/cibersortx:/src/data \
           -v $(pwd)/data/deconvolution/cibersortx:/src/outdir \
           cibersortx/fractions \
           --username <your-registered-email> \
           --token <your-token> \
           --mixture mixture_TPM.txt \
           --rmbatchBmode TRUE
```

(LM22 is the default signature matrix - no `--sigmatrix` flag needed unless swapping it out, per notebook 04's still-open "custom vs. default signature" decision.)

### Step 3 (not yet run): load results once produced

In [3]:
results_path = os.path.join(ciber_dir, "CIBERSORTx_Results.txt")
if os.path.exists(results_path):
    ciber_results = pd.read_csv(results_path, sep="\t")
    display(ciber_results.head())
else:
    print("CIBERSORTx_Results.txt not found yet - run Step 2 first.")

CIBERSORTx_Results.txt not found yet - run Step 2 first.
